In [ ]:
import os, sys

from gymnasium.wrappers import TimeLimit
import gymnasium as gym
from pathlib import Path

os.environ['PYTHONPATH'] = '/your/path/to/main/'
sys.path.append('/your/path/to/main/')
from merlion.algo_ray.rllib.algorithms.maml.merlion_main_general import MERLION, MERLIONConfig
from merlion.algo_ray.rllib.algorithms.maml.callback_save_population import _find_archive , MERLIONWithSaveCallbacks

 #v2.3.1 #use algo from local repo
import ray
from ray import air, tune
from ray.tune.registry import register_env
from gymnasium.wrappers import TimeLimit
from ray.air import Checkpoint
from ray.rllib.algorithms.algorithm import Algorithm

import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

import tempfile
import pickle
import json

In [ ]:
from ray.tune.registry import register_env
from merlion._non_sc.envs.merlion_env_combine import ScalarizedMetaEnv
import ray

ENV_NAME = [
    "mo-halfcheetah-v4",
    "deep-sea-treasure-v0",
    "resource-gathering-v0",
    "mo-mountaincar-v0",
    "mo-mountaincarcontinuous-v0",
    "four-room-v0",
    "mo-highway-fast-v0",
    "mo-reacher-v4",
    "mo-hopper-v4",
    "water-reservoir-v0",
]

ray.shutdown()
ray.init(ignore_reinit_error=True)

# ===== Option 1: Default argument (RECOMMENDED) =====
for env_id in ENV_NAME:
    register_env(
        f"Scalarized-{env_id}",
        lambda cfg=None, _env_id=env_id: ScalarizedMetaEnv({**(cfg or {}), "base_id": _env_id})
    )

In [ ]:
import psutil

# Get memory details
ram_info = psutil.virtual_memory()

# Display RAM information in GB
print(f"Total RAM: {ram_info.total / 1024 / 1024 / 1024:.2f} GB")
print(f"Available RAM: {ram_info.available / 1024 / 1024 / 1024:.2f} GB")
print(f"Used RAM: {ram_info.used / 1024 / 1024 / 1024:.2f} GB")
print(f"RAM Usage Percentage: {ram_info.percent}%")

In [ ]:
os.listdir()

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
def save_full_checkpoints_for_population(algo, out_dir: str):
    archive = _find_archive(algo)
    if archive is None:
        print("[SavePopulation] MERLION archive not found on algorithm; skipping.")
        return

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    pol = algo.get_policy()
    P = int(getattr(archive, "population_size", len(getattr(archive, "archive", []))))

    for i in range(P):
        item = archive.get_archive_item(i)
        if item is None:
            continue

        theta_i = item["theta"]
        w_i = np.asarray(item["weight"], dtype=np.float32)   # <- correct key

        # save the full RLlib checkpoint for policy i
        pol.set_weights(theta_i)
        ckpt_dir = out / f"checkpoint_policy_{i}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        path = algo.save(str(ckpt_dir))
        print(f"[SavePopulation] Saved RLlib checkpoint for i={i} → {path}")

        # save the associated weight next to it
        with open(ckpt_dir / "weight.json", "w") as f:
            json.dump([float(x) for x in w_i.ravel().tolist()], f)

# CHANGE THE NUM_OBJECTIVES PARAMETER MANUALLY

In [ ]:
rollout_fragment_length = 4

id1 = 2
print(f'running-Scalarized-{ENV_NAME[id1]}')

config = (
    MERLIONConfig()
    .rollouts(
        num_rollout_workers=4, rollout_fragment_length=rollout_fragment_length,
    )
    .framework("torch")
    .environment(f'Scalarized-{ENV_NAME[id1]}',
                 env_config = {"max_episode_steps": 10},
                 # disable_env_checking = False,
                 clip_actions=True,
                )
    .training(
        inner_adaptation_steps=1,
        meta_batch_size=1,
        maml_optimizer_steps=2,
        gamma=0.99,
        lambda_=1.0,
        lr=0.001,
        vf_loss_coeff=0.5,
        inner_lr=0.03,
        use_meta_env=True,
        clip_param=0.3,
        kl_target=0.01,
        kl_coeff=0.001,
        model=dict(fcnet_hiddens=[64, 64]),
        train_batch_size=rollout_fragment_length,
        
        #MERLION specific parameters
        population_size = 10, #default
        num_objectives = 3, ### must get manually 
    )
    .evaluation(
        evaluation_num_workers=1,
        evaluation_interval=10,
        enable_async_evaluation=True,
    )
    .callbacks(MERLIONWithSaveCallbacks)
)

In [ ]:
local_dir = "your/path/to/main/merlion/_non_sc/training"

num_iterations = 3
results = {}
runs = 1

# ensure base directory exists
os.makedirs(local_dir, exist_ok=True)

for k in range(runs):
    print(f"RUNNING NUMBER-{k}")

    # pre-create the experiment directory (avoids .tmp_checkpoint path errors)
    exp_dir = os.path.join(local_dir, f"merlion_{ENV_NAME[id1]}_{k}")
    os.makedirs(exp_dir, exist_ok=True)

    tuner = tune.Tuner(
        MERLION,
        param_space=config.to_dict(),
        run_config=air.RunConfig(
            name=f"merlion_hc_{k}",
            local_dir=local_dir,  # keep as-is
            stop={"training_iteration": num_iterations},
            failure_config=air.FailureConfig(fail_fast="raise"),  # keep as-is
            # explicit log files (avoids odd __stdout_file__ fields)
            log_to_file=True,
            # ("stdout.log", "stderr.log"),
            checkpoint_config=air.CheckpointConfig(
                checkpoint_at_end=True,
                num_to_keep=1
            ),
        )
    )

    results[k] = tuner.fit()

    # save the best checkpoit for each run
    best = results[k].get_best_result()  # (optionally pass metric=..., mode=...)
    if best.checkpoint is None:
        print(f"[WARN] Best result for run {k} has no checkpoint; skipping export.")
        continue

    algo = Algorithm.from_checkpoint(best.checkpoint)

    out_dir = os.path.join(local_dir, f"metapolicies-{k}")
    os.makedirs(out_dir, exist_ok=True)
    save_full_checkpoints_for_population(algo, out_dir)
    algo.stop()

## Fine-tuning Phase using SB3

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.ppo.policies import MlpPolicy
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

try:
    from merlion.algo_ray.rllib.algorithms.maml import merlion_finetuning_general_sb3 as ft
except:
    from algo_ray.rllib.algorithms.maml import merlion_finetuning_general_sb3 as ft

In [ ]:
from _non_sc.envs.merlion_ft_half_cheetah import ScalarizedMetaEnv

# (Re)start Ray cleanly
ray.shutdown()
ray.init(ignore_reinit_error=True, include_dashboard=False)

def make_env_ray(config):
    """
    RLlib factory: receives a dict `config` and must return a *Gymnasium Env instance*.
    We wrap with TimeLimit here so both RLlib and Gym share the same horizon behavior.
    """
    base = ScalarizedMetaEnv(**config)            # pass config keys into your env
    max_steps = int(config.get("max_steps", 10))
    return TimeLimit(base, max_episode_steps=max_steps)

register_env("MetaHC-Test", make_env_ray)
register_env("MoMetaHC-v0", lambda cfg: merlion_env_half_cheetah.make_env(cfg)) # register meta-env

In [ ]:
# Optional: re-register safely if re-running cells
try:
    gym.spec("MetaHC-Test")  # raises if not registered
    # Gymnasium 0.29.x: remove existing spec to allow re-register
    del gym.envs.registration.registry["MetaHC-Test"]
except Exception:
    pass

def make_env_gym(**config):
    base = ScalarizedMetaEnv(dict(ENV_CONF))
    max_steps = int(config.get("max_steps", 10))
    return TimeLimit(base, max_episode_steps=max_steps)

gym.register(
    id="MetaHC-Test",
    entry_point=make_env_gym,   # callable that returns an Env
)

In [ ]:
# === config ===
ENV_CONF = dict(base_id="mo-halfcheetah-v4", reward_dim=2, max_episode_steps=10)
RUNS = 3
PPO_ITERS = 100
NUM_WORKERS = 4
FRAG_LEN = 200

runs = 1
for k in range(runs):
    start_time = time.time()
    ft.finetune_with_local_perturbations(
        pop_dir=f"/mnt/hum01-home01/p88346bn/test/project/merlion/script/merlion/_non_sc/training/metapolicies-{k}",
        env_id="MetaHC-Test",
        out_dir=f"/mnt/hum01-home01/p88346bn/test/project/merlion/script/merlion/_non_sc/training/finetuning_hc_sb3-{k}",
        total_steps=5, # default 5000
        record_every=1, # default 1
        eval_episodes=1, # default 50
        m_perturb=5,
        eps=0.05,
        seed=7,
    )
    end_time = time.time()
    print(f"fine-tuning run-{k}:",end_time - start_time,"seconds")

In [ ]:
from merlion._non_sc.envs.merlion_env_combine import ScalarizedMetaEnv
import ray

ENV_NAME = [
    "mo-halfcheetah-v4",
    "deep-sea-treasure-v0",
    "resource-gathering-v0",
    "mo-mountaincar-v0",
    "mo-mountaincarcontinuous-v0",
    "four-room-v0",
    "mo-highway-fast-v0",
    "mo-reacher-v4",
    "mo-hopper-v4",
    "water-reservoir-v0",
]

ray.shutdown()
ray.init(ignore_reinit_error=True)

# ===== Opsi 1: Default argument (RECOMMENDED) =====
# Gunakan default argument agar Python capture nilai _env_id di saat registration
for env_id in ENV_NAME:
    register_env(
        f"Scalarized-{env_id}",
        lambda cfg=None, _env_id=env_id: ScalarizedMetaEnv({**(cfg or {}), "base_id": _env_id})
    )

In [ ]:
# === config ===
id1 = 2
ENV_CONF = dict(base_id=ENV_NAME[id1], reward_dim=3, max_episode_steps=10)
RUNS = 3
PPO_ITERS = 100
NUM_WORKERS = 4
FRAG_LEN = 200

In [ ]:
# Optional: re-register safely if re-running cells

def make_env_gym(**config):
    base = ScalarizedMetaEnv(dict(ENV_CONF))
    max_steps = int(config.get("max_steps", 10))
    return TimeLimit(base, max_episode_steps=max_steps)

gym.register(
    id=f"{env_id}-test",
    entry_point=make_env_gym,   # callable that returns an Env
)

In [ ]:
runs = 1
for k in range(runs):
    start_time = time.time()
    ft.finetune_with_local_perturbations(
        pop_dir=f"/mnt/hum01-home01/p88346bn/test/project/merlion/script/merlion/_non_sc/training/{ENV_NAME[id1]}/metapolicies-{k}",
        env_id=f"{env_id}-test",
        out_dir=f"/mnt/hum01-home01/p88346bn/test/project/merlion/script/merlion/_non_sc/training/{ENV_NAME[id1]}/finetuning_hc_sb3-{k}",
        total_steps=5, # default 5000
        record_every=1, # default 1
        eval_episodes=1, # default 50
        m_perturb=5,
        eps=0.05,
        seed=7,
    )
    end_time = time.time()
    print(f"fine-tuning run-{k}:",end_time - start_time,"seconds")

In [ ]:
# -*- coding: utf-8 -*-
# Environment-agnostic fine-tuning over multiple MO-Gymnasium tasks
# Requires: mo-gymnasium, gymnasium, ray[rllib], torch

import os
import glob
import numpy as np
import ray
from typing import Dict

from ray.tune.registry import register_env
from ray.rllib.algorithms.ppo import PPOConfig
from merlion.algo_ray.rllib.algorithms.maml.maml_loc import MAMLConfigLoc

# Your wrapper env
from merlion._non_sc.envs.merlion_env_combine import ScalarizedMetaEnv

# ===== Targets =====
ENV_NAME = [
    "mo-halfcheetah-v4",
    "deep-sea-treasure-v0",
    "resource-gathering-v0",
    "mo-mountaincar-v0",
    "mo-mountaincarcontinuous-v0",
    "mo-lunar-lander-v2",
    "water-reservoir-v0",
]

id1 = 2

TARGETS = [ENV_NAME[id1]]  # subset if needed

# ===== General knobs =====
RUNS = 3
PPO_ITERS = 100
NUM_WORKERS = 4
FRAG_LEN = 200
k = 1

# Root for outputs and checkpoints
OUT_ROOT = f"/mnt/hum01-home01/p88346bn/test/project/merlion/script/merlion/_non_sc/training/"
SAVE_DIR = f"/mnt/hum01-home01/p88346bn/test/project/merlion/script/merlion/_non_sc/training/{ENV_NAME[id1]}/"
CKPT_TPL = f"/mnt/hum01-home01/p88346bn/test/project/merlion/script/merlion/_non_sc/training/{ENV_NAME[id1]}/metapolicies-{k}"


# ===== Utilities =====
def register_all_envs():
    # Registers Scalarized-{env_id} for every env in TARGETS
    for env_id in TARGETS:
        def _creator(env_ctx, _env_id=env_id):
            cfg = dict(env_ctx or {})
            cfg.setdefault("base_id", _env_id)
            return ScalarizedMetaEnv(cfg)
        register_env(f"Scalarized-{env_id}", _creator)


def build_weights(d_obj: int, n_grid_2d: int = 21, n_sample_nd: int = 32, seed: int = 0) -> np.ndarray:
    if d_obj == 2:
        ws = np.linspace(0.0, 1.0, n_grid_2d, dtype=np.float32)
        return np.stack([ws, 1.0 - ws], axis=1).astype(np.float32)
    rng = np.random.RandomState(seed)
    W = rng.dirichlet(np.ones(d_obj), size=n_sample_nd).astype(np.float32)
    return W


def set_weight_all_workers(algo, w, scalarization="weighted_sum"):
    def _apply(env):
        if hasattr(env, "set_scalarization"):
            env.set_scalarization(scalarization)
        env.set_weight(w)
    algo.workers.foreach_worker(lambda wk: wk.foreach_env(_apply))


def copy_model_weights(src_algo, dst_algo):
    src_sd = src_algo.get_policy().model.state_dict()
    missing, unexpected = dst_algo.get_policy().model.load_state_dict(src_sd, strict=False)
    if missing or unexpected:
        print("[warn] state_dict mismatch\n  missing:", missing, "\n  unexpected:", unexpected)


def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p


def latest_checkpoint_or_none(parent_dir: str) -> str | None:
    # Find the newest leaf checkpoint_XXXX directory under a parent
    cands = sorted(glob.glob(os.path.join(parent_dir, "checkpoint_*")))
    return cands[-1] if cands else None


# Fresh cluster for a clean registry
ray.shutdown()
ray.init(ignore_reinit_error=True, include_dashboard=False)

register_all_envs()

for env_id in TARGETS:
    # Per-environment config; wrapper infers reward_dim
    ENV_CONF: Dict = dict(
        base_id=env_id,
        max_episode_steps=1000,
    )

    # Output dirs
    SAVE_DIR = ensure_dir(os.path.join(OUT_ROOT, env_id))

    # Probe reward dimension (for weight grid)
    probe_env = ScalarizedMetaEnv(dict(ENV_CONF))
    d_obj = int(getattr(probe_env, "d_obj", 2))
    WEIGHTS = build_weights(d_obj, n_grid_2d=21, n_sample_nd=32, seed=0)
    probe_env.close() if hasattr(probe_env, "close") else None

    for i in range(RUNS):
        print(f"\n=== [{env_id}] Fine-tuning run {i} ===")

        # Re-register in this fresh cluster
        register_all_envs()
        registered_name = f"Scalarized-{env_id}"

        # 2) Build PPO for fine-tuning
        ppo = (
            PPOConfig()
            .framework("torch")
            .environment(registered_name, clip_actions=True, env_config=ENV_CONF)
            .rollouts(num_rollout_workers=NUM_WORKERS, rollout_fragment_length=FRAG_LEN)
            .training(
                gamma=0.99, lambda_=0.95, lr=3e-4,
                model={"fcnet_hiddens": [64, 64], "fcnet_activation": "tanh"},
                train_batch_size=FRAG_LEN * max(1, NUM_WORKERS),
                sgd_minibatch_size=min(256, FRAG_LEN * max(1, NUM_WORKERS)),
                num_sgd_iter=10,
                clip_param=0.2, vf_loss_coeff=0.5, grad_clip=0.5,
            ).evaluation(
                evaluation_interval=1,            # memaksa buat eval worker set
                evaluation_num_workers=1,
                evaluation_config={"explore": False}
            )
            .build()
        )

        ckpt_dir = ensure_dir(os.path.join(SAVE_DIR, f"metapolicies-{k}/checkpoint_policy_0/checkpoint_000000"))
        ppo.restore(ckpt_dir)  # atau ppo.from_checkpoint(...)
        # lanjut training atau evaluasi
        res = ppo.evaluate()
        print(res)
        ppo.stop()

ray.shutdown()
print("\nALL FINE-TUNING DONE.")

In [ ]:
runs = 1
for k in range(runs):
    start_time = time.time()
    ft.finetune_with_local_perturbations(
        pop_dir=f"/mnt/hum01-home01/p88346bn/test/project/merlion/script/merlion/_non_sc/training/metapolicies-{k}",
        env_id="MetaHC-Test",
        out_dir=f"/mnt/hum01-home01/p88346bn/test/project/merlion/script/merlion/_non_sc/training/finetuning_hc_sb3-{k}",
        total_steps=5, # default 5000
        record_every=1, # default 1
        eval_episodes=1, # default 50
        m_perturb=5,
        eps=0.05,
        seed=7,
    )
    end_time = time.time()
    print(f"fine-tuning run-{k}:",end_time - start_time,"seconds")

## Print actions

In [ ]:
MODEL_PATH = "/mnt/hum01-home01/p88346bn/test/project/merlion/script/merlion/_non_sc/training/finetuning_hc_sb3-0/i000_m-1/sb3_model.zip"  # <- your file
ENV_ID     = "MetaHC-Test"
CSV_OUT    = "/mnt/hum01-home01/p88346bn/test/project/merlion/script/merlion/_non_sc/training/finetuning_hc_sb3-0/i000_m-1/action_plan.csv"

env = gym.make(ENV_ID, max_steps=10)
model = PPO.load(MODEL_PATH, env=env)

# Try to obtain human-readable action labels, otherwise use a[i]
action_labels = None
ue = getattr(env, "unwrapped", env)
for attr in ("get_action_names", "action_names", "action_labels"):
    if hasattr(ue, attr):
        names = getattr(ue, attr)
        action_labels = names() if callable(names) else names
        break

obs, info = env.reset(seed=7)
rows = []
t = 0
while True:
    action, _ = model.predict(obs, deterministic=True)
    action = np.array(action).ravel()

    row = {"t": t}
    if action_labels and len(action_labels) == action.size:
        for name, val in zip(action_labels, action):
            row[name] = float(val)
    else:
        for i, val in enumerate(action):
            row[f"a{i}"] = float(val)

    # Optional: capture reward vector if the env provides it
    mo = info.get("mo_reward")
    if mo is not None and len(mo) >= 3:
        row["profit"]    = float(mo[0])
        row["emission"]  = float(mo[1])
        row["inequality"]= float(mo[2])

    rows.append(row)

    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break
    t += 1

env.close()

df = pd.DataFrame(rows)
df.to_csv(CSV_OUT, index=False)
print(f"Saved action plan to {CSV_OUT}")